# exp02b — Activation Probe on Colleague's 229 Pairs

**Goal:** run exp01's probe pipeline on the same 229 valid pairs used by exp02 (logit lens),
enabling a direct method comparison on identical data.

**Data source:** `valid_pairs.json` from colleague's Google Drive cache (`stego_exp01/`).
Each entry contains pre-stored token ids (`stego_ids`, `stego_plen`, `open_ids`, `open_plen`)
generated with `meta-llama/Llama-3.1-8B-Instruct` — no re-generation needed.

**Comparison after this notebook:**

| Method | Source | Key metric |
|---|---|---|
| Logit lens (exp02) | colleague | onset layer, P(letter) curve |
| Activation probe (exp02b) | this notebook | per-layer AUROC, recall@1%FPR |

Same model, same pairs, complementary readouts.

In [ ]:
import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REAL_MODEL = 'meta-llama/Llama-3.1-8B-Instruct'
COLLEAGUE_CACHE = '/content/drive/MyDrive/stego_exp01'  # exp02's Google Drive cache

if IN_COLAB:
    if not os.path.exists('stego_CoT'):
        os.system('git clone -q https://github.com/annareshetnyak799-netizen/stego_CoT.git')
    os.chdir('stego_CoT')
    os.system('git pull -q')
    os.system('pip install -q -r requirements.txt')
    from google.colab import userdata, drive
    drive.mount('/content/drive')
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
else:
    COLLEAGUE_CACHE = 'notebooks/results/exp02_colleague'  # local fallback
    if os.path.exists('.env'):
        for line in open('.env'):
            if '=' in line and not line.startswith('#'):
                k, v = line.strip().split('=', 1)
                os.environ.setdefault(k, v)

print(f'Environment: {"Colab" if IN_COLAB else "Local"} | model: {REAL_MODEL}')
print(f'Cache dir: {COLLEAGUE_CACHE}')

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(42)
print('Imports OK')

## 1. Load colleague's valid_pairs.json

In [ ]:
PAIRS_PATH = f'{COLLEAGUE_CACHE}/valid_pairs.json'
assert os.path.exists(PAIRS_PATH), (
    f'{PAIRS_PATH} not found.\n'
    'Make sure Google Drive is mounted and colleague shared the stego_exp01 folder.')

with open(PAIRS_PATH) as f:
    valid_pairs = json.load(f)

required = {'stego_ids', 'stego_plen', 'open_ids', 'open_plen', 'payload'}
assert required.issubset(valid_pairs[0].keys()), (
    f'Missing fields in valid_pairs. Found: {list(valid_pairs[0].keys())}')

print(f'Loaded {len(valid_pairs)} valid pairs')
print(f'Payload examples: {[p["payload"] for p in valid_pairs[:5]]}')
print(f'Stego seq len example: {len(valid_pairs[0]["stego_ids"])} tokens '
      f'(prompt={valid_pairs[0]["stego_plen"]})')

## 2. Load model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def load_model(model_name=REAL_MODEL):
    hf_token = os.environ.get('HF_TOKEN')
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    dtype = torch.bfloat16 if device == 'cuda' else torch.float32
    tok = AutoTokenizer.from_pretrained(model_name, token=hf_token)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    if device == 'cuda':
        model = AutoModelForCausalLM.from_pretrained(
            model_name, dtype=dtype, device_map='auto', token=hf_token).eval()
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_name, dtype=dtype, token=hf_token).to(device).eval()
    print(f'Model loaded on {device} ({dtype})')
    print(f'n_layers={model.config.num_hidden_layers} | hidden={model.config.hidden_size}')
    return model, tok, device

model, tok, device = load_model()

## 3. Extract activations from pre-stored token ids

No re-generation needed: `stego_ids` already contain the full sequence (prompt + generated tokens).
We run a single forward pass per sample with `output_hidden_states=True`.

Three readout points:
- **pre**: last prompt token (index `plen - 1`)
- **mean**: mean over CoT tokens (indices `plen:)`)
- **post**: last generated token (index `-1`)

In [ ]:
ACT_CACHE = f'{COLLEAGUE_CACHE}/probe_activations.npz'

@torch.no_grad()
def extract_three_readouts(ids_list, plen, device):
    """Single forward pass → (n_layers+1, 3, d_model) for pre/mean/post."""
    ids = torch.tensor(ids_list, device=device, dtype=torch.long).unsqueeze(0)
    out = model(input_ids=ids, output_hidden_states=True)
    pre, mean_act, post = [], [], []
    for h in out.hidden_states:          # tuple of (1, seq, d)
        h = h[0]                         # (seq, d)
        pre.append(h[plen - 1].float().cpu().numpy())
        cot = h[plen:]
        mean_act.append(cot.mean(0).float().cpu().numpy() if len(cot) > 0
                        else h[-1].float().cpu().numpy())
        post.append(h[-1].float().cpu().numpy())
    return np.stack(pre), np.stack(mean_act), np.stack(post)  # each: (n_layers+1, d)


def collect_activations(pairs, label):
    """Returns dict: pre/mean/post each (N, n_layers+1, d_model)."""
    key = 'stego' if label == 'stego' else 'open'
    all_pre, all_mean, all_post = [], [], []
    for i, p in enumerate(pairs):
        pre, mean_act, post = extract_three_readouts(
            p[f'{key}_ids'], p[f'{key}_plen'], device)
        all_pre.append(pre)
        all_mean.append(mean_act)
        all_post.append(post)
        if (i + 1) % 20 == 0:
            print(f'  {label} {i+1}/{len(pairs)}')
    return {
        'pre':  np.stack(all_pre),
        'mean': np.stack(all_mean),
        'post': np.stack(all_post),
    }


if os.path.exists(ACT_CACHE):
    d = dict(np.load(ACT_CACHE))
    A_stego = {k: d[f'stego_{k}'] for k in ('pre', 'mean', 'post')}
    A_open  = {k: d[f'open_{k}']  for k in ('pre', 'mean', 'post')}
    print('Loaded activations from cache:', {k: v.shape for k, v in A_stego.items()})
else:
    print(f'Extracting activations for {len(valid_pairs)} pairs...')
    A_stego = collect_activations(valid_pairs, 'stego')
    A_open  = collect_activations(valid_pairs, 'open')
    np.savez(ACT_CACHE,
             stego_pre=A_stego['pre'],   stego_mean=A_stego['mean'], stego_post=A_stego['post'],
             open_pre=A_open['pre'],     open_mean=A_open['mean'],   open_post=A_open['post'])
    print('Saved:', ACT_CACHE)

print('Shapes:', {k: v.shape for k, v in A_stego.items()})

## 4. Per-layer probes

In [ ]:
def recall_at_fpr(y, scores, fpr=0.01):
    thr = np.quantile(scores[y == 0], 1 - fpr)
    return (scores[y == 1] > thr).mean()

def per_layer_probe(As, Ao, key):
    X = np.concatenate([As[key], Ao[key]], 0)   # (2N, n_layers, d)
    y = np.array([1]*len(As[key]) + [0]*len(Ao[key]))
    nL = X.shape[1]
    cv = min(5, int((y==0).sum()), int((y==1).sum()))
    rows = []
    for L in range(nL):
        XL = StandardScaler().fit_transform(X[:, L, :])
        clf = LogisticRegression(max_iter=2000, C=1.0)
        auroc = cross_val_score(clf, XL, y, cv=cv, scoring='roc_auc').mean()
        clf.fit(XL, y)
        sc = clf.predict_proba(XL)[:, 1]
        rows.append((L, auroc, recall_at_fpr(y, sc)))
    return np.array(rows)

results = {}
for key in ['pre', 'mean', 'post']:
    r = per_layer_probe(A_stego, A_open, key)
    best_l = int(r[np.argmax(r[:, 1]), 0])
    best_a = r[:, 1].max()
    results[key] = {'rows': r, 'best_layer': best_l, 'best_auroc': best_a}
    print(f'{key:>5} | best layer {best_l:>2} | AUROC {best_a:.3f} '
          f'| recall@1%FPR {r[best_l, 2]:.2f}')

## 5. Layer profile plot

In [ ]:
SAVE_DIR = f'{COLLEAGUE_CACHE}'
os.makedirs(SAVE_DIR, exist_ok=True)

colors = {'pre': '#f59e0b', 'mean': '#2563eb', 'post': '#7c3aed'}
fig, ax = plt.subplots(figsize=(9, 4.5))
for key in ['pre', 'mean', 'post']:
    r = results[key]['rows']
    ax.plot(r[:, 0], r[:, 1], 'o-', color=colors[key], label=f'{key} (best={results[key]["best_auroc"]:.3f})')
ax.axhline(0.5, ls=':', color='gray', label='chance')
# Mark logit-lens onset layer from exp02 (layer 29)
ax.axvline(29, ls='--', color='#dc2626', alpha=0.6, label='logit-lens onset (exp02, layer 29)')
ax.set_ylim(0.4, 1.02)
ax.set_xlabel('layer (residual-stream index)')
ax.set_ylabel('AUROC')
ax.set_title('exp02b: activation probe on 229 pairs (Llama-3.1-8B)\nvs exp02 logit-lens onset layer')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/probe_layer_profile.png', dpi=120)
plt.show()

## 6. Comparison table: probe vs logit lens

Logit lens results from exp02 (colleague):
- stego peak P = 0.970 at layer 32
- half-peak onset = layer 29
- open peak P = 0.037

In [ ]:
# Logit lens results from exp02 (hardcoded from colleague's summary.json)
logitlens = {
    'stego_peak_prob':  0.970,
    'stego_peak_layer': 32,
    'onset_layer':      29,
    'open_peak_prob':   0.037,
}

print('=' * 60)
print('METHOD COMPARISON — same 229 pairs, same model (Llama-3.1-8B)')
print('=' * 60)
print()
print('ACTIVATION PROBE (exp02b):')
for key in ['pre', 'mean', 'post']:
    r = results[key]
    print(f'  {key:>5}: best layer {r["best_layer"]:>2}, AUROC {r["best_auroc"]:.3f}')
print()
print('LOGIT LENS (exp02, colleague):')
print(f'  stego peak P(letter) = {logitlens["stego_peak_prob"]:.3f} at layer {logitlens["stego_peak_layer"]}')
print(f'  half-peak onset      = layer {logitlens["onset_layer"]} (~{logitlens["onset_layer"]/32:.0%} depth)')
print(f'  open baseline P      = {logitlens["open_peak_prob"]:.3f}')
print()
print('KEY QUESTION: does probe peak layer align with logit-lens onset?')
probe_post_best = results['post']['best_layer']
delta = abs(probe_post_best - logitlens['onset_layer'])
print(f'  post-gen probe best layer = {probe_post_best}')
print(f'  logit-lens onset layer    = {logitlens["onset_layer"]}')
print(f'  delta = {delta} layers  -->',
      'ALIGNED' if delta <= 3 else 'DIVERGE — investigate')

## 7. Transfer test (held-out split)

In [ ]:
key = 'mean'
best_l = results[key]['best_layer']
X = np.concatenate([A_stego[key][:, best_l, :], A_open[key][:, best_l, :]], 0)
y = np.array([1]*len(A_stego[key]) + [0]*len(A_open[key]))
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
sc = StandardScaler().fit(Xtr)
clf = LogisticRegression(max_iter=2000).fit(sc.transform(Xtr), ytr)
tr_auroc = roc_auc_score(yte, clf.predict_proba(sc.transform(Xte))[:, 1])
print(f'Held-out split AUROC (mean-CoT, layer {best_l}): {tr_auroc:.3f}')
print('Note: for real transfer, train on arithmetic tasks, test on factual QA.')

## 8. Save results

In [ ]:
import json as _json

summary = {
    'model': REAL_MODEL,
    'n_pairs': len(valid_pairs),
    'probe': {
        k: {'best_layer': int(results[k]['best_layer']),
            'best_auroc': float(results[k]['best_auroc'])}
        for k in ['pre', 'mean', 'post']
    },
    'transfer_auroc_held_out': float(tr_auroc),
    'logitlens_ref': logitlens,
}
out_path = f'{SAVE_DIR}/probe_summary.json'
with open(out_path, 'w') as f:
    _json.dump(summary, f, indent=2)
print(_json.dumps(summary, indent=2))
print(f'Saved -> {out_path}')